# 爱尔法鲁酒吧问题
## El Farol Bar Problem

**背景**

1994 年，经济学家 W. Brian Arthur 提出了这个思想实验：

圣菲（Santa Fe）有一家叫「爱尔法鲁」的酒吧，每周四晚上会有现场音乐演出。  
对于镇上的 100 个居民来说：
- 如果去酒吧的人**少于 60 人**，那晚上就很愉快，去了划算
- 如果去酒吧的人**超过 60 人**，就会非常拥挤，不去反而更好

**难题在于**：每个人只能根据过去的历史来预测本周会有多少人去，然后各自独立决定。
没有人互相商量，没有协调机制，也没有任何中央规划。

---

**为什么这个问题有趣？**

它打破了传统经济学的假设。传统模型会说：如果所有人都是「理性的」，他们会共同推导出一个均衡解。
但这里有一个悖论：

> 假设所有人都预测「本周会有 70 人去」，于是大家都选择不去 → 实际只有 0 人去 → 这个预测是错的  
> 假设所有人都预测「本周会有 30 人去」，于是大家都去 → 实际有 100 人去 → 这个预测也是错的

不存在一个「所有人都同意」的预测。多样化的、不协调的预测反而是这个系统运作的必要条件。

**涌现的神奇之处**：尽管每个人的决策互相干扰、没有任何协调，整个系统的出勤人数会自发地在 60 人附近徘徊——宏观秩序从微观的混乱中涌现出来。

## 模型设计

Arthur 的原始模型：

1. **N 个 Agent**：每个人代表一个居民（默认 100 人）
2. **策略库（Strategy Pool）**：从一个公共策略池中，每个 agent 随机抽取 k 个策略
3. **策略是什么**：一个策略就是一个预测函数——输入过去若干周的出勤历史，输出对本周出勤人数的预测
4. **策略评估**：每个 agent 跟踪自己所有策略的「虚拟得分」（即使没有使用该策略，也会记录它的预测误差）
5. **决策规则**：agent 选用当前得分最高的策略。如果该策略预测人数 < 阈值，就去；否则不去
6. **更新**：每周结束后，所有策略根据实际出勤人数更新得分

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 策略池

我们定义一个多样化的策略池，每种策略都是基于历史数据的不同预测方式：

In [ ]:
def build_strategy_pool(N: int) -> list:
    """
    构建策略池。每个策略是一个函数：history (list) -> predicted_attendance (float)
    history 是过去若干周的实际出勤人数列表（从旧到新）
    """
    strategies = []

    # --- 类型 1：固定预测 ---
    # 无论历史如何，总是预测某个固定值
    for fixed_val in [10, 20, 30, 40, 50, 60, 70, 80, 90]:
        val = fixed_val  # 闭包捕获
        strategies.append((f"固定预测={val}", lambda h, v=val: v))

    # --- 类型 2：滞后预测（直接复制 k 周前的出勤数）---
    for lag in [1, 2, 3, 4]:
        strategies.append((f"滞后{lag}周", lambda h, k=lag: h[-k] if len(h) >= k else h[-1]))

    # --- 类型 3：移动平均 ---
    for window in [2, 3, 4, 6, 8, 12]:
        strategies.append((f"均值{window}周", lambda h, w=window: np.mean(h[-w:])))

    # --- 类型 4：镜像预测（N - 上周） ---
    # 逻辑：如果上周很拥挤，大家这周不会去；反之亦然
    strategies.append(("镜像上周", lambda h, n=N: n - h[-1]))
    strategies.append(("镜像均值4", lambda h, n=N: n - np.mean(h[-4:])))

    # --- 类型 5：线性趋势外推 ---
    # 根据最近两周的变化趋势预测
    strategies.append(("线性趋势", lambda h: 2 * h[-1] - h[-2] if len(h) >= 2 else h[-1]))
    strategies.append(("趋势+均值", lambda h: 0.5 * (2 * h[-1] - h[-2]) + 0.5 * np.mean(h[-4:]) if len(h) >= 4 else np.mean(h)))

    # --- 类型 6：周期性预测 ---
    # 检测 2 周周期（隔周交替）
    strategies.append(("2周周期", lambda h: h[-2] if len(h) >= 2 else h[-1]))
    strategies.append(("3周周期", lambda h: h[-3] if len(h) >= 3 else h[-1]))

    # --- 类型 7：加权移动平均（越近越重要）---
    def weighted_avg(h, w=4):
        segment = h[-w:]
        weights = np.arange(1, len(segment) + 1, dtype=float)
        return np.dot(segment, weights) / weights.sum()

    strategies.append(("加权均值4", lambda h: weighted_avg(h, 4)))
    strategies.append(("加权均值8", lambda h: weighted_avg(h, 8)))

    # --- 类型 8：中位数 ---
    strategies.append(("中位数4", lambda h: np.median(h[-4:])))
    strategies.append(("中位数8", lambda h: np.median(h[-8:])))

    return strategies


pool = build_strategy_pool(100)
print(f"策略池共有 {len(pool)} 种策略：")
for name, _ in pool:
    print(f"  · {name}")

## Agent 与模拟器

In [ ]:
class Agent:
    """
    一个居民。持有 k 个随机策略，始终使用当前得分最高的那个做决策。
    """

    def __init__(self, strategies: list):
        self.strategies = strategies  # list of (name, func)
        # 每个策略的累积得分（越高越好，用负的均方误差累积）
        self.scores = np.zeros(len(strategies))

    def predict(self, history: list) -> float:
        """用当前最佳策略预测本周出勤人数"""
        best_idx = np.argmax(self.scores)
        return self.strategies[best_idx][1](history)

    def decide(self, history: list, threshold: int) -> bool:
        """根据预测决定本周是否去酒吧"""
        prediction = self.predict(history)
        return prediction < threshold  # 预测人少于阈值 → 去

    def update(self, history: list, actual: int):
        """用实际出勤数更新所有策略的得分（虚拟评估）"""
        for i, (_, func) in enumerate(self.strategies):
            prediction = func(history)
            error = (prediction - actual) ** 2
            self.scores[i] -= error  # 误差越大，得分越低


class ElFarolSimulation:

    def __init__(self, N: int = 100, threshold: int = 60,
                 k: int = 5, warmup_weeks: int = 10, seed: int = 42):
        """
        N          : 居民总数
        threshold  : 酒吧舒适容量
        k          : 每个 agent 持有的策略数量
        warmup_weeks: 用于初始化历史记录的随机周数
        """
        self.N = N
        self.threshold = threshold
        self.k = k
        rng = np.random.default_rng(seed)

        pool = build_strategy_pool(N)

        # 初始化 agent，每人随机抽取 k 个不重复策略
        self.agents = []
        for _ in range(N):
            indices = rng.choice(len(pool), size=k, replace=False)
            agent_strategies = [pool[i] for i in indices]
            self.agents.append(Agent(agent_strategies))

        # 用随机数据预热历史（提供足够的历史让策略能开始运作）
        self.history = list(rng.integers(10, 90, size=warmup_weeks))
        self.attendance_log = []  # 正式记录从 run() 开始

    def step(self):
        """模拟一周"""
        decisions = [agent.decide(self.history, self.threshold) for agent in self.agents]
        actual_attendance = sum(decisions)

        # 先更新历史，再让 agent 学习
        for agent in self.agents:
            agent.update(self.history, actual_attendance)

        self.history.append(actual_attendance)
        self.attendance_log.append(actual_attendance)
        return actual_attendance

    def run(self, weeks: int = 200):
        for _ in range(weeks):
            self.step()
        return np.array(self.attendance_log)

## 运行模拟

In [ ]:
sim = ElFarolSimulation(N=100, threshold=60, k=5, seed=42)
attendance = sim.run(weeks=300)

print(f"前 50 周平均出勤率：{attendance[:50].mean():.1f} / 100")
print(f"后 50 周平均出勤率：{attendance[-50:].mean():.1f} / 100")
print(f"全程标准差：{attendance.std():.2f}")
print(f"超过阈值的周数：{(attendance > 60).sum()} / {len(attendance)}")

## 可视化：涌现的均衡

In [ ]:
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']

fig = plt.figure(figsize=(14, 9))
gs = GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

weeks = np.arange(1, len(attendance) + 1)

# --- 图1：完整时间序列 ---
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(weeks, attendance, color='#4e79a7', alpha=0.7, linewidth=0.9, label='每周出勤人数')

# 滚动均值（窗口=20周）
window = 20
rolling_mean = np.convolve(attendance, np.ones(window)/window, mode='valid')
ax1.plot(weeks[window-1:], rolling_mean, color='#f28e2b', linewidth=2, label=f'{window}周滚动均值')

ax1.axhline(60, color='#e15759', linewidth=1.5, linestyle='--', label='阈值 = 60')
ax1.fill_between(weeks, attendance, 60,
                 where=(attendance < 60), alpha=0.08, color='green', label='出勤 < 阈值')
ax1.fill_between(weeks, attendance, 60,
                 where=(attendance >= 60), alpha=0.08, color='red', label='出勤 ≥ 阈值')

ax1.set_xlabel('周数')
ax1.set_ylabel('出勤人数')
ax1.set_title('爱尔法鲁酒吧问题：出勤人数时间序列', fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)
ax1.set_ylim(0, 100)
ax1.yaxis.set_major_locator(ticker.MultipleLocator(20))
ax1.grid(axis='y', linestyle=':', alpha=0.5)

# --- 图2：出勤分布直方图 ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(attendance, bins=20, color='#4e79a7', edgecolor='white', alpha=0.85)
ax2.axvline(60, color='#e15759', linewidth=1.5, linestyle='--', label='阈值 = 60')
ax2.axvline(attendance.mean(), color='#f28e2b', linewidth=1.5, linestyle='-',
            label=f'均值 = {attendance.mean():.1f}')
ax2.set_xlabel('出勤人数')
ax2.set_ylabel('频次')
ax2.set_title('出勤人数分布')
ax2.legend(fontsize=9)
ax2.grid(axis='y', linestyle=':', alpha=0.5)

# --- 图3：滚动均值收敛图 ---
ax3 = fig.add_subplot(gs[1, 1])
windows = [5, 10, 20, 50]
colors = ['#76b7b2', '#59a14f', '#f28e2b', '#e15759']
for w, c in zip(windows, colors):
    rm = np.convolve(attendance, np.ones(w)/w, mode='valid')
    ax3.plot(weeks[w-1:], rm, color=c, linewidth=1.4, alpha=0.85, label=f'{w}周')
ax3.axhline(60, color='black', linewidth=1, linestyle='--', alpha=0.6, label='阈值 = 60')
ax3.set_xlabel('周数')
ax3.set_ylabel('滚动均值')
ax3.set_title('不同窗口的滚动均值收敛趋势')
ax3.legend(fontsize=9, title='窗口大小')
ax3.set_ylim(0, 100)
ax3.grid(axis='y', linestyle=':', alpha=0.5)

plt.suptitle('N=100, 阈值=60, 每人持有 5 个策略', y=1.01, fontsize=10, color='gray')
plt.show()

## 参数探索：改变策略数量 k

每个 agent 持有的策略数量 `k` 影响他们的「适应能力」。k 越大，agent 越灵活，但整个系统会如何变化？

In [ ]:
k_values = [1, 2, 5, 10, 15]
weeks_to_run = 500
results = {}

for k in k_values:
    sim = ElFarolSimulation(N=100, threshold=60, k=k, seed=42)
    results[k] = sim.run(weeks=weeks_to_run)

fig, axes = plt.subplots(len(k_values), 1, figsize=(13, 10), sharex=True)
colors = ['#e15759', '#f28e2b', '#4e79a7', '#59a14f', '#b07aa1']

for ax, (k, att), color in zip(axes, results.items(), colors):
    ax.plot(att, color=color, alpha=0.6, linewidth=0.8)
    window = 30
    rm = np.convolve(att, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(att)), rm, color=color, linewidth=2)
    ax.axhline(60, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax.set_ylim(0, 100)
    ax.set_ylabel(f'k={k}')
    ax.text(0.99, 0.85, f'μ={att.mean():.1f}  σ={att.std():.1f}',
            transform=ax.transAxes, ha='right', fontsize=9, color='gray')
    ax.grid(axis='y', linestyle=':', alpha=0.4)

axes[-1].set_xlabel('周数')
fig.suptitle('不同策略数量 k 对涌现均衡的影响\n（粗线为30周滚动均值，虚线为阈值60）',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n各 k 值统计摘要：")
print(f"{'k':>4} | {'均值':>6} | {'标准差':>6} | {'超过阈值比例':>10}")
print("-" * 38)
for k, att in results.items():
    print(f"{k:>4} | {att.mean():>6.1f} | {att.std():>6.2f} | {(att>60).mean()*100:>9.1f}%")

## 参数探索：改变阈值

阈值决定酒吧「舒适容量」占总人口的比例。系统是否总能收敛到这个阈值附近？

In [ ]:
thresholds = [20, 40, 60, 80]
weeks_to_run = 400

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=True)
axes = axes.flatten()
colors = ['#e15759', '#f28e2b', '#4e79a7', '#59a14f']

for ax, threshold, color in zip(axes, thresholds, colors):
    sim = ElFarolSimulation(N=100, threshold=threshold, k=5, seed=42)
    att = sim.run(weeks=weeks_to_run)

    ax.plot(att, color=color, alpha=0.5, linewidth=0.8)
    window = 20
    rm = np.convolve(att, np.ones(window)/window, mode='valid')
    ax.plot(np.arange(window-1, len(att)), rm, color=color, linewidth=2, label='20周滚动均值')
    ax.axhline(threshold, color='black', linewidth=1.5, linestyle='--', label=f'阈值={threshold}')

    ax.set_title(f'阈值 = {threshold}  (μ={att[-100:].mean():.1f})', fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle=':', alpha=0.4)
    ax.set_xlabel('周数')
    ax.set_ylabel('出勤人数')

fig.suptitle('不同阈值下的涌现均衡（k=5, N=100）', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 观察策略的「进化」

在模拟过程中，哪些类型的策略会被更多使用？

In [ ]:
sim = ElFarolSimulation(N=100, threshold=60, k=5, seed=42)
sim.run(weeks=300)

# 统计每个 agent 当前最佳策略的类型
best_strategy_names = []
for agent in sim.agents:
    best_idx = np.argmax(agent.scores)
    name = agent.strategies[best_idx][0]
    # 提取策略大类
    if '固定' in name:
        category = '固定预测'
    elif '均值' in name or '加权' in name or '中位数' in name:
        category = '均值/平滑'
    elif '滞后' in name or '周期' in name:
        category = '滞后/周期'
    elif '镜像' in name:
        category = '镜像预测'
    elif '趋势' in name:
        category = '趋势外推'
    else:
        category = '其他'
    best_strategy_names.append(category)

# 统计分布
from collections import Counter
counts = Counter(best_strategy_names)
categories = list(counts.keys())
values = [counts[c] for c in categories]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(categories, values, color='#4e79a7', edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val} 人', va='center', fontsize=10)
ax.set_xlabel('持有该策略作为最佳策略的 agent 数量')
ax.set_title('300周后：各策略类型的「市场份额"', fontweight='bold')
ax.grid(axis='x', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## 小结

### 我们观察到了什么？

1. **自发涌现的均衡**：没有任何协调机制，出勤人数会在阈值附近自组织。这个均衡不是某个主体设计出来的，它从数百次独立决策的相互作用中涌现。

2. **多样性是必要条件**：当 `k=1`（每人只有一个策略）时，系统可能振荡剧烈。策略的多样性反而是整体稳定的来源。

3. **阈值的自我实现**：无论阈值设为 20、40、60 还是 80，系统总能「找到」那个目标并徘徊在附近。这说明涌现均衡是阈值的函数，而非偶然。